In [8]:
import ast, warnings
import numpy as np
import pandas as pd
import json, re, joblib, datetime as dt
import statsmodels
from pathlib import Path
import xgboost as xgb
import sys
from pathlib import Path
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
warnings.filterwarnings("ignore")

In [3]:
HORIZON, M, SEED = 7, 7, 42

def cari_root(penanda=("data", "processed")):
    mulai = Path.cwd().resolve()
    for p in (mulai, *mulai.parents):
        if p.joinpath(*penanda).exists():
            return p
    raise FileNotFoundError("Folder data/processed tidak ditemukan")

ROOT = cari_root()
DIR_DATA = ROOT / "data" / "processed" / "forecasting"
DIR_TABEL = ROOT / "reports" / "tables" / "forecasting"

In [4]:
# Data
deret = pd.read_csv(DIR_DATA / "deret_waktu_harian.csv", parse_dates=["tanggal"])
kalender = pd.read_csv(DIR_DATA / "kalender_fitur.csv", parse_dates=["tanggal"])
harga_seri = {n: g.sort_values("tanggal").set_index("tanggal")["harga"].astype(float)
              for n, g in deret.groupby("komoditas")}
kal = kalender.set_index("tanggal").select_dtypes(include=["number", "bool"]).astype(float)
LAGS = [1, 7, 14, 28]

In [5]:
# Parameter final dari notebook backtest
PF = pd.read_csv(DIR_TABEL / "parameter_final.csv", index_col=0)
for c in ["order", "seasonal", "params"]:
    PF[c] = PF[c].map(lambda v: ast.literal_eval(v) if isinstance(v, str) else None)

# Fungsi peramalan
def prediksi_naive(y, h=HORIZON):
    return np.repeat(y.iloc[-1], h)

def ramal_arima(y, order, h=HORIZON):
    trend = "c" if order[1] == 0 else "n"
    return np.asarray(ARIMA(y.asfreq("D"), order=order, trend=trend).fit().forecast(h))

def ramal_sarima(y, order, seasonal, h=HORIZON):
    trend = "c" if order[1] == 0 and seasonal[1] == 0 else "n"
    model = SARIMAX(y.asfreq("D"), order=order, seasonal_order=(*seasonal, M), trend=trend,
                    enforce_stationarity=False, enforce_invertibility=False)
    return np.asarray(model.fit(disp=False, maxiter=200).forecast(h))

def buat_data(y):
    F = pd.DataFrame({f"sel_{k}": y - y.shift(k) for k in LAGS})
    F["dev_m7"], F["dev_m28"] = y - y.rolling(7).mean(), y - y.rolling(28).mean()
    F["std_7"] = y.rolling(7).std()
    bagian = []
    for h in range(1, HORIZON + 1):
        d = F.assign(h=h, tgl_target=F.index + pd.Timedelta(days=h), target=y.shift(-h) - y)
        bagian.append(d.join(kal, on="tgl_target"))
    return pd.concat(bagian)

def ramal_xgb(D, y, akhir, params):
    fitur = [c for c in D.columns if c not in ("tgl_target", "target")]
    latih = D[D["tgl_target"] <= akhir].dropna()
    opsi = {"objective": "reg:absoluteerror", "eta": 0.05, "subsample": 0.8, "seed": SEED,
            "max_depth": params["max_depth"]}
    model = xgb.train(opsi, xgb.DMatrix(latih[fitur], label=latih["target"]),
                      num_boost_round=params["n_estimators"])
    asal = D.loc[akhir].sort_values("h")
    return model.predict(xgb.DMatrix(asal[fitur])) + y.loc[akhir]

PF[["model", "order", "seasonal", "params"]]

,model,order,seasonal,params
Bawang Bombay,naive,None,None,None
Bayam (Jenis Sayur),naive,None,None,None
Buncis,sarima,"(0, 0, 1)","(0, 1, 1)",None
Kacang Kedelai (Kualitas Lokal),sarima,"(0, 1, 0)","(0, 1, 0)",None
Kangkung (Jenis Panjang),naive,None,None,None
Kubis,arima,"(0, 1, 0)",None,None
Sawi (Jenis Hijau),xgboost,None,None,"{'max_depth': 2, 'n_estimators': 300}"
Terong Ungu (Jenis Sayur),naive,None,None,None
Tomat,naive,None,None,None
Wortel (Kualitas Lokal),xgboost,None,None,"{'max_depth': 2, 'n_estimators': 100}"


In [6]:
# Ramalan final: fit ulang pada seluruh data, parameter terkunci, ramal HORIZON hari ke depan
MODEL_FINAL = PF["model"].to_dict()     
baris = []
for nama, y in harga_seri.items():
    m, r = MODEL_FINAL[nama], PF.loc[nama]
    if m == "naive":    pred = prediksi_naive(y)
    elif m == "arima":  pred = ramal_arima(y, r["order"])
    elif m == "sarima": pred = ramal_sarima(y, r["order"], r["seasonal"])
    else:               pred = ramal_xgb(buat_data(y), y, y.index[-1], r["params"])

    tgl = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=HORIZON)
    baris += [{"komoditas": nama, "model": m, "tanggal": t, "h": i + 1, "ramalan": round(float(p))}
              for i, (t, p) in enumerate(zip(tgl, pred))]

ramalan_final = pd.DataFrame(baris)
ramalan_final.to_csv(DIR_TABEL / "ramalan_final.csv", index=False)
ramalan_final.pivot(index="komoditas", columns="h", values="ramalan")

h,1,2,3,4,5,6,7
komoditas,,,,,,,
Bawang Bombay,31429,31429,31429,31429,31429,31429,31429
Bayam (Jenis Sayur),2250,2250,2250,2250,2250,2250,2250
Buncis,17201,17005,16516,16490,16966,17591,17452
Kacang Kedelai (Kualitas Lokal),15750,14250,14250,14250,15750,15750,14250
Kangkung (Jenis Panjang),1833,1833,1833,1833,1833,1833,1833
Kubis,8500,8500,8500,8500,8500,8500,8500
Sawi (Jenis Hijau),2750,2750,2750,2750,2750,2750,2750
Terong Ungu (Jenis Sayur),9667,9667,9667,9667,9667,9667,9667
Tomat,9667,9667,9667,9667,9667,9667,9667


In [8]:
# Simpan model dan metadata 
DIR_MODEL = ROOT / "models" / "forecasting"
DIR_MODEL.mkdir(parents=True, exist_ok=True)

banding = pd.read_csv(DIR_TABEL / "perbandingan_model.csv", index_col=0)
uji = pd.read_csv(DIR_TABEL / "evaluasi_test.csv", index_col=0)
f = lambda v: None if pd.isna(v) else round(float(v), 3)

registry = {}
for nama, y in harga_seri.items():
    m, r = MODEL_FINAL[nama], PF.loc[nama]
    slug = re.sub(r"\W+", "_", nama.lower()).strip("_")
    berkas = None

    if m == "arima":
        trend = "c" if r["order"][1] == 0 else "n"
        hasil = ARIMA(y.asfreq("D"), order=r["order"], trend=trend).fit()
        berkas = f"{slug}_arima.pkl"; hasil.save(DIR_MODEL / berkas)
    elif m == "sarima":
        trend = "c" if r["order"][1] == 0 and r["seasonal"][1] == 0 else "n"
        hasil = SARIMAX(y.asfreq("D"), order=r["order"], seasonal_order=(*r["seasonal"], M), trend=trend,
                        enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
        berkas = f"{slug}_sarima.pkl"; hasil.save(DIR_MODEL / berkas)
    elif m == "xgboost":
        D = buat_data(y)
        fitur = [c for c in D.columns if c not in ("tgl_target", "target")]
        latih = D[D["tgl_target"] <= y.index[-1]].dropna()
        booster = xgb.train({"objective": "reg:absoluteerror", "eta": 0.05, "subsample": 0.8,
                             "seed": SEED, "max_depth": r["params"]["max_depth"]},
                            xgb.DMatrix(latih[fitur], label=latih["target"]),
                            num_boost_round=r["params"]["n_estimators"])
        berkas = f"{slug}_xgboost.json"; booster.save_model(DIR_MODEL / berkas)

    registry[nama] = {
        "model": m,
        "parameter": {k: (list(r[k]) if isinstance(r[k], tuple) else r[k])
                      for k in ("order", "seasonal", "params") if isinstance(r[k], (tuple, dict))},
        "berkas_model": berkas,
        "fitur_xgboost": fitur if m == "xgboost" else None,
        "nilai_terakhir": float(y.iloc[-1]) if m == "naive" else None,
        "data_awal": str(y.index[0].date()), "data_akhir": str(y.index[-1].date()), "n_latih": len(y),
        "pemenang_validasi": banding.loc[nama, "pemenang"],
        "mae_validasi": f(banding.loc[nama, m]), "mae_naive_validasi": f(banding.loc[nama, "naive"]),
        "mae_test": f(uji.loc[nama, "mae"]), "mae_naive_test": f(uji.loc[nama, "mae_naive"]),
        "rasio_test": f(uji.loc[nama, "rasio_vs_naive"]),
    }

meta = {"dibuat": dt.datetime.now().isoformat(timespec="seconds"), "horizon": HORIZON, "seed": SEED,
        "lags_xgboost": LAGS, "periode_musiman": M,
        "versi": {"pandas": pd.__version__, "statsmodels": statsmodels.__version__, "xgboost": xgb.__version__},
        "komoditas": registry}
(DIR_MODEL / "metadata_model.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

pd.DataFrame(registry).T[["model", "berkas_model", "data_akhir", "mae_test", "rasio_test"]]

,model,berkas_model,data_akhir,mae_test,rasio_test
Bawang Bombay,naive,None,2026-08-31,606.143,1.0
Bayam (Jenis Sayur),naive,None,2026-08-31,56.536,1.0
Buncis,sarima,buncis_sarima.pkl,2026-08-31,764.589,1.511
Kacang Kedelai (Kualitas Lokal),sarima,kacang_kedelai_kualitas_lokal_sarima.pkl,2026-08-31,1017.857,1.462
Kangkung (Jenis Panjang),naive,None,2026-08-31,0.0,None
Kubis,arima,kubis_arima.pkl,2026-08-31,357.036,1.0
Sawi (Jenis Hijau),xgboost,sawi_jenis_hijau_xgboost.json,2026-08-31,108.929,1.0
Terong Ungu (Jenis Sayur),naive,None,2026-08-31,273.821,1.0
Tomat,naive,None,2026-08-31,416.857,1.0
Wortel (Kualitas Lokal),xgboost,wortel_kualitas_lokal_xgboost.json,2026-08-31,293.594,1.037


In [10]:
def cari_root(penanda=("data", "processed")):
    mulai = Path.cwd().resolve()
    for p in (mulai, *mulai.parents):
        if p.joinpath(*penanda).exists():
            return p
    raise FileNotFoundError("Folder data/processed tidak ditemukan")

ROOT = cari_root()
DIR_TABEL = ROOT / "reports" / "tables" / "forecasting"
sys.path.insert(0, str(ROOT))

from src.forecasting import ramal_semua

baru = ramal_semua()
lama = pd.read_csv(DIR_TABEL / "ramalan_final.csv", parse_dates=["tanggal"])
selisih = baru["ramalan"].values - lama["ramalan"].values
print("Selisih maksimum:", np.abs(selisih).max())

Selisih maksimum: 0.0
